# S01E04 — Send It

**Zadanie:** Wypełnić deklarację transportu w Systemie Przesyłek Konduktorskich (SPK)  
i przesłać ją do `/verify`. Dane są z dokumentacji na Hub-ie — część w plikach graficznych (vision).

**Dane przesyłki:**
- Nadawca: `450202122`, z Gdańska do Żarnowca
- Waga: 2800 kg, Zawartość: kasety z paliwem do reaktora
- Budżet: 0 PP (darmowe lub finansowane przez System), bez uwag specjalnych

**Techniki:** multimodalność (vision), web scraping, agent iteracyjny

In [ ]:
import os, re, base64, requests
from urllib.parse import urljoin
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
DOC_BASE   = "https://hub.ag3nts.org/dane/doc/"

print("Konfiguracja OK.")

In [ ]:
def fetch_text(url: str) -> str:
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text

def fetch_image_b64(url: str) -> tuple[str, str]:
    """Zwraca (base64, media_type)."""
    resp = requests.get(url)
    resp.raise_for_status()
    ext = url.split(".")[-1].lower()
    media_type = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg",
                  "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/png")
    return base64.b64encode(resp.content).decode(), media_type

def extract_links(md_text: str, base_url: str) -> list[str]:
    """Wyciągnij linki z markdown (relative i absolute)."""
    links = re.findall(r'\[.*?\]\((.*?)\)', md_text)
    result = []
    for link in links:
        if link.startswith("http"):
            result.append(link)
        else:
            result.append(urljoin(base_url, link))
    return result


# Pobierz główny plik dokumentacji
index_url  = DOC_BASE + "index.md"
index_text = fetch_text(index_url)
print(f"index.md ({len(index_text)} znaków):")
print(index_text[:800])

In [ ]:
# Pobierz wszystkie linkowane pliki
linked_urls = extract_links(index_text, DOC_BASE)
print(f"Linki w index.md: {linked_urls}")

docs_text   = {"index.md": index_text}  # tekstowe pliki
docs_images = {}                         # pliki graficzne

image_exts = {".png", ".jpg", ".jpeg", ".gif", ".webp"}

for url in linked_urls:
    name = url.split("/")[-1]
    ext  = os.path.splitext(name)[1].lower()
    try:
        if ext in image_exts:
            b64, mt = fetch_image_b64(url)
            docs_images[name] = (b64, mt, url)
            print(f"  [obraz] {name}")
        else:
            docs_text[name] = fetch_text(url)
            print(f"  [tekst] {name} ({len(docs_text[name])} znaków)")
    except Exception as e:
        print(f"  [błąd] {name}: {e}")

print(f"\nTeksty: {list(docs_text.keys())}")
print(f"Obrazy: {list(docs_images.keys())}")

In [ ]:
# Przetwórz pliki graficzne modelem vision
image_descriptions = {}

for name, (b64, mt, url) in docs_images.items():
    resp = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=2000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "base64", "media_type": mt, "data": b64}
                },
                {
                    "type": "text",
                    "text": "Przepisz DOKŁADNIE całą zawartość tego dokumentu. Zachowaj wszystkie pola, wartości, formatowanie i strukturę tabeli. To jest dokumentacja systemu SPK."
                }
            ]
        }]
    )
    image_descriptions[name] = resp.content[0].text
    print(f"[{name}] przepisano {len(image_descriptions[name])} znaków")
    print(image_descriptions[name][:500])
    print()

In [ ]:
# Połącz całą dokumentację i poproś Claude o wypełnienie formularza
all_docs = "\n\n".join(
    [f"=== {name} ===\n{text}" for name, text in docs_text.items()] +
    [f"=== {name} (z obrazu) ===\n{desc}" for name, desc in image_descriptions.items()]
)

SHIPMENT_DATA = """
Nadawca (identyfikator): 450202122
Punkt nadawczy: Gdańsk
Punkt docelowy: Żarnowiec
Waga: 2800 kg
Zawartość: kasety z paliwem do reaktora
Budżet: 0 PP (przesyłka ma być darmowa lub finansowana przez System)
Uwagi specjalne: BRAK — nie dodawaj żadnych uwag
"""

resp = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Na podstawie dokumentacji SPK wypełnij deklarację transportu.

DOKUMENTACJA:
{all_docs}

DANE PRZESYŁKI:
{SHIPMENT_DATA}

Instrukcje:
1. Znajdź w dokumentacji wzór deklaracji i wypełnij go dokładnie.
2. Zachowaj oryginalne formatowanie, separatory i kolejność pól ze wzoru.
3. Ustal poprawny kod trasy Gdańsk-Żarnowiec z siatki połączeń.
4. Wybierz kategorię przesyłki finansowaną przez System (budżet 0 PP).
5. Zwróć TYLKO wypełniony formularz — bez żadnego dodatkowego komentarza.
"""
    }]
)

declaration = resp.content[0].text
print("Wypełniona deklaracja:")
print(declaration)

In [ ]:
# Wyślij deklarację do Hub-u
payload = {
    "apikey": API_KEY,
    "task": "sendit",
    "answer": {"declaration": declaration}
}
resp = requests.post(VERIFY_URL, json=payload)
result = resp.json()
print(result)

# Jeśli błąd — popraw deklarację na podstawie feedbacku
if result.get("code") != 0:
    print("\nFeedback Hub-u:", result.get("message"))
    print("Popraw deklarację i wyślij ponownie.")